In [5]:
import pandas as pd
import numpy as np
import duckdb

np.random.seed(7)
n = 400

categories = np.random.choice(['Electronics', 'Clothing', 'Home', 'Sports'], n, p=[0.35, 0.25, 0.25, 0.15])
regions    = np.random.choice(['North', 'South', 'East', 'West'], n)
channels   = np.random.choice(['Online', 'In-store', 'Phone', None], n, p=[0.40, 0.35, 0.15, 0.10])
quantity   = np.random.randint(1, 8, n)
unit_price = np.where(
    np.random.rand(n) < 0.18, np.nan,
    np.random.choice([29.99, 49.99, 99.99, 149.99, 299.99, 499.99, 799.99], n)
)

df_sales = pd.DataFrame({
    'sale_id':    range(1, n + 1),
    'category':   categories,
    'region':     regions,
    'channel':    channels,
    'quantity':   quantity,
    'unit_price': unit_price,
})
df_sales['revenue'] = df_sales['quantity'] * df_sales['unit_price']

df_categories = pd.DataFrame({
    'category':   ['Electronics', 'Clothing', 'Home', 'Sports'],
    'department': ['Technology', 'Fashion', 'Living', 'Active Lifestyle'],
    'buyer':      ['Ana Silva', 'João Costa', 'Maria Pinto', 'Rui Ferreira'],
})
print("Data generated successfully!")

Data generated successfully!


In [6]:
print(round(df_sales['unit_price'].mean(), 2))

print(round(df_sales['revenue'].isnull().mean() * 100, 1))

262.03
21.5


In [7]:
query = """
SELECT 
    c.department, 
    SUM(s.revenue) AS total_revenue
FROM df_sales s
INNER JOIN df_categories c ON s.category = c.category
WHERE s.channel = 'In-store'
GROUP BY c.department
ORDER BY total_revenue DESC
"""
res = duckdb.sql(query).df()
print(res)

res.to_csv('results.csv', index=False)
print("\nExported to results.csv successfully!")

         department  total_revenue
0        Technology       37738.55
1           Fashion       33538.87
2            Living       21029.18
3  Active Lifestyle       19439.32

Exported to results.csv successfully!


In [8]:
def channel_filter(df, channel):
    return df[df['channel'] == channel].copy()

print(channel_filter(df_sales, 'In-store').head())

    sale_id     category region   channel  quantity  unit_price  revenue
2         3     Clothing   West  In-store         6      299.99  1799.94
3         4         Home  North  In-store         4      299.99  1199.96
7         8  Electronics  South  In-store         1         NaN      NaN
11       12         Home   East  In-store         3         NaN      NaN
13       14  Electronics   East  In-store         5      799.99  3999.95
